# ETL Cleaning — Geolocation Table
**Source:** `olist_geolocation_dataset.csv`  
**Output:** `data/cleaned/geolocation_cleaned.csv`

### Key characteristics of this table
- Each zip code prefix can have many coordinate records (average 52 per zip code)
- This table will be used as a JOIN key with customers and sellers tables
- A JOIN key must be unique, so we aggregate to one row per zip code prefix

### Cleaning Steps
1. Load raw data
2. Initial inspection
3. Check duplicate zip code prefixes
4. Aggregate by zip code prefix
5. Export cleaned data

## Step 1 Load Raw Data

In [1]:
import pandas as pd

# ── 1. Load raw data ──────────────────────────────────────────
df = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

## Step 2 Initial Inspection

In [2]:
# ── 2. Initial inspection ─────────────────────────────────────
print("=== Shape ===")
print(df.shape)

print("\n=== Column names ===")
print(df.columns.tolist())

print("\n=== First 5 rows ===")
print(df.head())

print("\n=== Current dtypes ===")
print(df.dtypes)

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Numeric columns summary ===")
print(df.describe())

print("\n=== Categorical columns summary ===")
print(df.describe(include='str'))

=== Shape ===
(1000163, 5)

=== Column names ===
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

=== First 5 rows ===
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   
2                         1046       -23.546129       -46.642951   
3                         1041       -23.544392       -46.639499   
4                         1035       -23.541578       -46.641607   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  

=== Current dtypes ===
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                  

## Step 3 Check Duplicate Zip Code Prefixes

This table will be used as a JOIN key with customers and sellers.  
A JOIN key must be unique per row, otherwise each customer would multiply into 52 rows after JOIN.  
We check how many coordinate records exist per zip code prefix.  

In [3]:
# ── 3. Check duplicate zip codes ─────────────────────────────
print(f"Total rows: {len(df)}")
print(f"Unique zip_code_prefix: {df['geolocation_zip_code_prefix'].nunique()}")
print(f"\nDuplicate zip_code_prefix: {df.duplicated(subset=['geolocation_zip_code_prefix']).sum()}")

Total rows: 1000163
Unique zip_code_prefix: 19015

Duplicate zip_code_prefix: 981148


## Step 4 Aggregate by Zip Code Prefix

Each zip code prefix is compressed into one representative row.  
Latitude and longitude are averaged to approximate the center of the zip code area.  
City and state are taken from the first occurrence.

In [4]:
# ── 4. Aggregate by zip_code_prefix (take mean lat/lng) ───────
df_geo = df.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
).reset_index()

print(f"Rows before: {len(df)}")
print(f"Rows after aggregation: {len(df_geo)}")

Rows before: 1000163
Rows after aggregation: 19015


## Step 5 Export Cleaned Data

In [5]:
# ── 5. Export cleaned data ────────────────────────────────────
df_geo.to_csv('../data/cleaned/geolocation_cleaned.csv', index=False)

print(f"Exported: {len(df_geo)} rows")
print("Saved to: data/cleaned/geolocation_cleaned.csv")

Exported: 19015 rows
Saved to: data/cleaned/geolocation_cleaned.csv
